# Classificação de Texto com Redes Neurais Recorrentes (RNN)

As Redes Neurais Recorrentes (RNN) são arquiteturas projetadas para processar dados sequenciais, como textos, séries temporais ou áudio. Diferente de redes tradicionais, as RNNs possuem conexões recorrentes que permitem manter um estado interno (memória) sobre o que já foi processado na sequência.

Neste exemplo, construiremos um classificador de sentimentos simples para identificar se frases curtas são **Positivas (1)** ou **Negativas (0)** utilizando o módulo `SimpleRNN` do TensorFlow/Keras.

In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

frases = [
    "eu amei este produto",
    "maravilhoso e muito bom",
    "excelente atendimento",
    "gostei bastante do resultado",
    "horrivel nao gostei",
    "pessimo produto muito ruim",
    "odiei o servico péssimo",
    "ruim e atrasou bastante"
]

sentimentos = np.array([1, 1, 1, 1, 0, 0, 0, 0])

tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(frases)
vocab_size = len(tokenizer.word_index) + 1

sequencias = tokenizer.texts_to_sequences(frases)
sequencias_pad = pad_sequences(sequencias, padding="post")

## Estrutura do Modelo

O modelo consiste em três camadas principais:
1. **Embedding**: Converte as palavras representadas por inteiros em vetores densos de tamanho fixo.
2. **SimpleRNN**: A camada recorrente básica que processa a sequência palavra por palavra, atualizando seu estado interno.
3. **Dense**: Uma camada de saída com ativação *sigmoid* para realizar a classificação binária (positivo ou negativo).

In [4]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=8),
    SimpleRNN(units=16),
    Dense(units=1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(sequencias_pad, sentimentos, epochs=30, verbose=0)

## Testando o Modelo

Agora vamos passar novas frases para testar a capacidade de generalização do modelo treinado.

In [5]:
novas_frases = [
    "este servico é muito bom",
    "pessimo e ruim"
]

novas_sequencias = tokenizer.texts_to_sequences(novas_frases)
novas_sequencias_pad = pad_sequences(novas_sequencias, maxlen=sequencias_pad.shape[1], padding="post")

predicoes = model.predict(novas_sequencias_pad)

for i, frase in enumerate(novas_frases):
    sentimento = "Positivo" if predicoes[i][0] > 0.5 else "Negativo"
    print(f"Frase: '{frase}' -> Sentimento: {sentimento} ({predicoes[i][0]:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
Frase: 'este servico é muito bom' -> Sentimento: Positivo (0.53)
Frase: 'pessimo e ruim' -> Sentimento: Negativo (0.47)
